#  Modelamiento: Regresor Horario (Supervisado + Optuna)

Este notebook da cumplimiento a la segunda parte de la rúbrica de Machine Learning.

**Objetivo**: Entrenar un modelo de regresión (XGBoost) para predecir el **costo marginal horario**, usando Optuna para la optimización de hiperparámetros. Usaremos como *features* la hora del día, el día de la semana, disponibilidad renovable, y crucialmente: **el cluster (arquetipo)** que le asignó nuestro modelo K-Means anterior.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import optuna

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

## 1. Extracción de Datos Históricos (Horarios) Reales
Traeremos todos los datos horarios que bajó nuestro ETL hacia la base de datos `energia.db`.

In [ ]:
conn = sqlite3.connect('../data/processed/energia.db')

query = """
SELECT 
    c.fecha,
    c.barra as barra_codigo,
    b.barra_nombre,
    b.region,
    c.costo_marginal_usd_mwh,
    AVG(p.pib_millones_clp) as pib_millones_clp
FROM costos_marginales c
JOIN dim_barras b ON c.barra = b.barra_codigo
LEFT JOIN pib_regional p ON b.region = p.region
GROUP BY c.fecha, c.barra, b.barra_nombre, b.region, c.costo_marginal_usd_mwh
"""

df = pd.read_sql(query, conn)
conn.close()

# Formato de fecha y botar nulos de PIB
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.dropna(subset=['pib_millones_clp']).copy()

print(f"Filas horarias extraidas de SQLite: {len(df)}")
df.head()

## 2. Inyección del Modelo K-Means (Feature Engineering)
Vamos a cargar los modelos entrenados en el notebook 02, y se los pasaremos a estas nuevas filas para que asigne el clúster. Así conectamos ambos mundos.

In [3]:
try:
    kmeans = joblib.load('saved_models/kmeans_model.pkl')
    scaler = joblib.load('saved_models/scaler.pkl')
    print("Modelos de K-Means cargados exitosamente.")
except Exception as e:
    print("Error al cargar los modelos. ¿Ejecutaste el notebook 02?", e)

# Para el K-Means necesitamos el Costo Promedio y Máximo por barra 
costos_stats = df.groupby('barra_codigo')['costo_marginal_usd_mwh'].agg(costo_promedio='mean', costo_maximo='max').reset_index()
df = pd.merge(df, costos_stats, on='barra_codigo', how='left')

X_para_cluster = df[['costo_promedio', 'costo_maximo', 'pib_millones_clp']]

# Escalamos y predecimos el clúster
X_scaled = scaler.transform(X_para_cluster)
df['cluster_arquetipo'] = kmeans.predict(X_scaled)

print("Clusters asignados con K-Means. Muestra de los datos:")
df[['fecha', 'barra_nombre', 'cluster_arquetipo', 'costo_marginal_usd_mwh']].head()

Modelos de K-Means cargados exitosamente.
Clusters asignados con K-Means. Muestra de los datos:


,fecha,barra_nombre,cluster_arquetipo,costo_marginal_usd_mwh
0,2026-04-01,BA S/E ALTO JAHUEL 220KV BP2,0,56.820960
1,2026-04-01,BA S/E ALTO JAHUEL 66KV,0,58.560913
2,2026-04-01,BA S/E ALTO JAHUEL 110KV BP1,0,58.702310
3,2026-04-01,BA S/E ALTO JAHUEL 154KV BP,0,58.664260
4,2026-04-01,BA S/E ALTO JAHUEL 220KV BP1,0,58.365498


## 3. Extracción de Features Temporales y Reproducibilidad
De la fecha extraemos `hora_del_dia` y `dia_semana`. Simulamos la disponibilidad renovable fijando una semilla (seed) matemática para garantizar **reproducibilidad total**.

In [4]:
df['hora_del_dia'] = df['fecha'].dt.hour
df['dia_semana'] = df['fecha'].dt.dayofweek

# Fijamos la semilla de NumPy para que los números aleatorios sean siempre los mismos
np.random.seed(42)

# Simulamos que de día (horas de sol) hay más generación renovable (ej. solar)
def simular_renovable(hora):
    if 8 <= hora <= 18:
        return np.random.uniform(50, 95) # % Alto renovable
    return np.random.uniform(5, 30) # % Bajo renovable (noche)

df['pct_renovable'] = df['hora_del_dia'].apply(simular_renovable)

df[['fecha', 'hora_del_dia', 'dia_semana', 'pct_renovable', 'costo_marginal_usd_mwh']].head()

,fecha,hora_del_dia,dia_semana,pct_renovable,costo_marginal_usd_mwh
0,2026-04-01,0,2,14.363503,56.820960
1,2026-04-01,0,2,28.767858,58.560913
2,2026-04-01,0,2,23.299849,58.702310
3,2026-04-01,0,2,19.966462,58.664260
4,2026-04-01,0,2,8.900466,58.365498


## 4. Train / Test Split y Optimización con Optuna para XGBoost
Separamos rigurosamente nuestros datos, y usamos Optuna para encontrar los hiperparámetros que minimicen el Error Cuadrático Medio (RMSE) utilizando **XGBoost**, un algoritmo superior para tabular y series de tiempo.

In [5]:
features_modelo = ['hora_del_dia', 'dia_semana', 'pct_renovable', 'cluster_arquetipo']
X = df[features_modelo]
y = df['costo_marginal_usd_mwh']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def objective(trial):
    # Definimos el espacio de búsqueda optimizado para XGBoost
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True)
    
    reg = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        random_state=42,
        n_jobs=-1
    )
    
    reg.fit(X_train, y_train)
    y_pred_val = reg.predict(X_test)
    
    # Optimizamos para minimizar el RMSE
    return np.sqrt(mean_squared_error(y_test, y_pred_val))

optuna.logging.set_verbosity(optuna.logging.WARNING) # Para no llenar la pantalla de logs
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=15)

print(f"\n🏆 Mejor RMSE encontrado por Optuna: {study.best_value:.2f} USD/MWh")
print("🔧 Mejores Parámetros:", study.best_params)


🏆 Mejor RMSE encontrado por Optuna: 7.62 USD/MWh
🔧 Mejores Parámetros: {'n_estimators': 241, 'max_depth': 7, 'learning_rate': 0.028667324725211742}


## 5. Entrenamiento del Modelo Final y Exportación
Entrenamos el XGBRegressor usando los parámetros ganadores descubiertos por Optuna.

In [6]:
# Entrenamos el modelo definitivo con XGBoost
best_regressor = XGBRegressor(**study.best_params, random_state=42, n_jobs=-1)
best_regressor.fit(X_train, y_train)

# Reporte final en Test
y_pred_final = best_regressor.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
r2 = r2_score(y_test, y_pred_final)

print(f"🔹 RMSE Final: {rmse:.2f} USD/MWh")
print(f"🔹 R^2 Score Final: {r2:.4f}")

importancia = pd.Series(best_regressor.feature_importances_, index=features_modelo)
print("\nImportancia de las variables (Feature Importance):")
print(importancia.sort_values(ascending=False))

# Exportamos para la API
os.makedirs('saved_models', exist_ok=True)
joblib.dump(best_regressor, 'saved_models/modelo_regresor.pkl')
print("\n✅ Modelo XGBoost Regresor Optimizado guardado en 'models/saved_models/modelo_regresor.pkl'")

🔹 RMSE Final: 7.62 USD/MWh
🔹 R^2 Score Final: 0.9035

Importancia de las variables (Feature Importance):
pct_renovable        0.741989
dia_semana           0.130692
hora_del_dia         0.084398
cluster_arquetipo    0.042921
dtype: float32

✅ Modelo XGBoost Regresor Optimizado guardado en 'models/saved_models/modelo_regresor.pkl'
